In [ ]:
# === BACKEND SETUP (qiskit_backend_config) ===
# -----------------------------------------------
# Common backend helper — works with Qiskit 2.x / qiskit-aer / qiskit-ibm-runtime.
# Provides drop-in replacements for the removed Aer, BasicAer, execute(), and IBMQ.
#
# HOW TO SELECT A BACKEND
# -----------------------
# Simulator (default, runs locally — no credentials needed):
#   BACKEND_NAME = "qasm_simulator"        # shot-based measurements
#   BACKEND_NAME = "statevector_simulator" # full statevector
#   BACKEND_NAME = "unitary_simulator"     # full unitary matrix
#
# Real IBM hardware (requires IBM Quantum account):
#   Set the environment variable:  IBM_QUANTUM_TOKEN=<your_token>
#   Then choose a device, e.g.:
#   BACKEND_NAME = "ibm_brisbane"
#   BACKEND_NAME = "least_busy"  # auto-pick least-busy device
#
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from qiskit_backend_config import (get_backend, execute, Aer, BasicAer,
                                   IBMQ, least_busy, job_monitor, assemble, combine)

# ── Choose your backend here ──────────────────────────────────────────────
BACKEND_NAME = "qasm_simulator"  # change to a device name for real hardware
# ── (Optional) IBM Quantum token — leave empty to use env var or saved creds
IBM_QUANTUM_TOKEN = os.environ.get("IBM_QUANTUM_TOKEN", "")

# Build the backend object used throughout this notebook
_token_arg = IBM_QUANTUM_TOKEN if IBM_QUANTUM_TOKEN else None
backend = get_backend(BACKEND_NAME, token=_token_arg)
print(f'Backend ready: {backend}')


In [ ]:
#Two-Qubit Gates
#Amitava Chakraborty - 13th Dec 2020
from qiskit_backend_config import IBMQ
# Loading your IBM Q account(s)
IBMQ.load_account()

While implementing controlled S and T gates in qiskit and we wonder whether there is any difference between using the [`cu1`][1] and [`crz`][2] gates for this purpose. Qiskit doesn't seem to include explicit versions of these two gates.

```
circuit = qiskit.QuantumCircuit(2)
# a)
circuit.crz(theta=math.pi/2, control_qubit=0, target_qubit=1)
# b)
circuit.cu1(theta=math.pi/2, control_qubit=0, target_qubit=1)
```
In terms of the unitary the `cu1` gate seems closer to what one would want. Are there any other consequences we should be aware of?
**Note : cu1 will be deprecated in favor of cp, but the final answer still holds. Using cp will be more concise, but the answer using Gate().control renders more nicely in diagrams (as controlled S not as controlled P (pi/2))**

In [ ]:
# Useful additional packages
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import math
# Importing standard Qiskit libraries and configuring account
from qiskit import QuantumCircuit, ClassicalRegister, QuantumRegister
from qiskit_backend_config import execute
from qiskit.visualization import circuit_drawer
from qiskit.visualization import plot_bloch_multivector, plot_histogram
from qiskit.quantum_info import state_fidelity
from qiskit_backend_config import BasicAer
from qiskit.circuit.library import HGate, RZGate, CXGate, CCXGate, XGate, SwapGate
from qiskit.circuit import ControlledGate

u_backend = BasicAer.get_backend('unitary_simulator')
s_backend = BasicAer.get_backend('statevector_simulator')
q_backend = BasicAer.get_backend('qasm_simulator')

In [ ]:
circuit = QuantumCircuit(2)
# a)
circuit.crz(theta=math.pi/2, control_qubit=0, target_qubit=1)
# b)
circuit.cp(theta=math.pi/2, control_qubit=0, target_qubit=1)
circuit.draw(output='mpl')

In [ ]:
job = execute(circuit, u_backend)
job.result().get_unitary(circuit, decimals=3)

In [ ]:
job = execute(circuit, s_backend)
plot_bloch_multivector (job.result().get_statevector(circuit))

We can see the gate matrices very easily in qiskit

In [ ]:
from qiskit.circuit.library import U1Gate
gate = U1Gate(math.pi/2)
print(gate.to_matrix()) 
gate = RZGate(math.pi/2)
print(gate.to_matrix())

Now let us investigate three issues that are relevant regarding these $U_1, R_z, CU_1$ and $CR_z$ gates

# Issue 1
A single qubit gate is ($\sigma_z$ is a Pauli-Z gate and $\beta = {\theta}/2$)

$$
e^{i\beta\sigma_z} =
\begin{bmatrix}
e^{i\beta} & 0 \\
0 & e^{-i\beta}
\end{bmatrix}
= e^{i\beta}
\begin{bmatrix}
1 & 0 \\
0 & e^{-i2\beta}
\end{bmatrix} = e^{i\beta} R_Z(-2\beta).
$$

However, we have seen the above gate is implemented using $U_1(2\beta)$, where
$
U_1(\lambda) = 
\begin{bmatrix}
1 & 0 \\
0 & e^{i\lambda}
\end{bmatrix}
$

So is $e^{i\beta} R_Z(-2\beta)$ equivalent to $U_1(2\beta)$?



# Issue 1 : Explanation
$R_z$ gate from M. Nielsen and I. Chuang textbook (page 174):


$$R_z(\theta) = e^{-i\theta Z/2} =\begin{pmatrix} e^{-i \theta /2} &0 \\ 0&e^{i \theta /2} \end{pmatrix}$$

If we use this definition for $R_z(\theta)$ then:

$$R_z(\theta) = e^{-i \theta/2} \begin{pmatrix} 1 &0 \\ 0&e^{i \theta} \end{pmatrix} = e^{-i \theta/2} U_1(\theta)$$

Therefore, $R_z(\theta)$ and $U_1(\theta)$ can be regarded as equivalent gates, because the global phase $e^{-i \theta/2}$ can be neglected. Note that this comparison is not true for their controlled versions: $cR_z(\theta)$ and $cU_1(\theta)$ can't be regarded as equivalent gates.

**Note : $e^{i\theta/2} R_Z(\theta) = U_1(\theta)$, so with $\frac{\theta}{2}=\beta$ we have $e^{i\beta} R_Z(2\beta) = U_1(2\beta)$. Note the difference: in this case the $Z$-rotation is positive, whereas in the original issue (issue 1) it is negative.**

# Issue 2 : Why a global phase is usually used when it does not really matter in quantum computation
By convention, we often write a single qubit gate as:

$$U=e^{i \alpha} R_z(\beta) R_y(\gamma) R_z(\delta)$$

We notice that in addition to the three rotations, there is a coefficient $e^{i \alpha}$. This extra phase $e^{i \alpha}$ shouldn't really matter as it will only add a global phase in the computation. So, why is it usually written ? Is it because we want to "mathematically" identify the expression of the unitary but in term of physics this phase will never be added in practice on a quantum computer ?



# Issue 2 : Explanation
## A reason why we need that $e^{i \alpha}$ term

It is right that the global phase $e^{i \alpha}$ will not change the action of the gate, but let's consider these two gates:

$$
U1\big(\frac{\pi}{2}\big) = 
\begin{pmatrix}
1 & 0 \\
0 & i
\end{pmatrix} 
\qquad 
R_z\big(\frac{\pi}{2}\big) = 
\begin{pmatrix}
e^{-i \frac{\pi}{4}} & 0 \\
0 & e^{i \frac{\pi}{4}}
\end{pmatrix}$$

It can be easily seen that $R_z\big(\frac{\pi}{2}\big) = e^{-i \frac{\pi}{4}} U1\big(\frac{\pi}{2}\big)$. So both gates are differ by a global phase $e^{-i \frac{\pi}{4}}$ which means that they are equavalent when we apply them in the circuits. 

Now let us investigate a particular issue. 
### Issue 3 : Why is a different convention used for the Rz implementation on IBM Q?

A $z$ rotation gate is defined as
$$
Rz(\theta)=\mathrm{e}^{-i\frac{\theta}{2}Z}=
\begin{pmatrix}
\mathrm{e}^{-i\frac{\theta}{2}} & 0 \\
0 & \mathrm{e}^{i\frac{\theta}{2}}
\end{pmatrix},
$$

however, when one uses $Rz$ gate on IBM Q, the results are these (tested in Visualization of state vector):

 - $Rz(\theta)|0\rangle = |0\rangle$
 - $Rz(\theta)|1\rangle = \mathrm{e}^{i\theta}|1\rangle$

This means that a matrix representation of such gate is

$$
Rz^{\text{IBM}}(\theta)=
\begin{pmatrix}
1 & 0 \\
0 & \mathrm{e}^{i\theta}
\end{pmatrix}.
$$

So, it seems that $Rz(\theta)$ is on IBM Q implemented with $U1(\theta)$ gate.

Since gates $Rz(\theta)$ and $Rz(\theta)^\text{IBM}$ differs only by global phase $\theta/2$ ($Rz^\text{IBM} = \mathrm{e}^{i\frac{\theta}{2}}Rz(\theta))$, this difference can be neglected. However, in case of controlled version of $z$ rotation the global phase matters. 

Controlled version of $Rz$ is described by matrix

$$
CRz(\theta)=
\begin{pmatrix}
1 & 0 & 0 & 0\\
0 & 1 & 0 & 0\\
0 & 0 & \mathrm{e}^{-i\frac{\theta}{2}} & 0 \\
0 & 0 & 0 & \mathrm{e}^{i\frac{\theta}{2}}
\end{pmatrix}
$$

On IBM Q, controlled version of $Rz$ behaves according to this matrix which is right.

Despite the global phase can be neglected in case of single qubit gates, one would expect that relation between a single qubit gate $U$ and its contolled version, i.e.

$$
CU=
\begin{pmatrix}
I & O \\
O & U
\end{pmatrix},
$$

will be preserved. In some cases the convention used by IBM for $Rz$ can be confusing.


### Issue 3 : The Explanation

[Qiskit's definition of $R_z$](https://github.com/Qiskit/qiskit-terra/blob/f72f5cd5fcdc83b323685b5e4fb1f0545fc67ebd/qiskit/circuit/library/standard_gates/rz.py#L40-L44) gate coincides with the conventional definition used, for example, in M. Nielsen and I. Chuang's textbook (page 174):

$$R_z(\theta) = \begin{pmatrix} e^{-i \theta/2}  &0 \\ 0&e^{i \theta/2} \end{pmatrix}$$

The "problem" is not in the **definitions**, the "problem" is in the **implementations**. 

Although Qiskit's implementation of `rz` gate coincides with the `u1` gate, the implementations of `crz` and `cu1` are different and coincide with the conventional definitions. Nevertheless, I think this still can cause some problems. For example, after creating a circuit with `rz` gate in it, one may want to create a controlled version of the circuit with Qiskit's `get_controlled_circuit` method that will not give `crz` gate, but `cu1`, as a result, there will be some logical errors.

Here are the codes and corresponding circuits in `qasm` format for all mentioned 4 gates

**$R_z$ gate:**
```
circuit_rz = QuantumCircuit(1)
circuit_rz.rz(1.4, 0)
circuit_rz = transpile(circuit_rz, basis_gates=['u1', 'u2', 'u3', 'cx'], optimization_level=0)
---------------------------------------------------------------------------------------------
u1(1.4) q[0];
```
**$cR_z$ gate:**
```
circuit_crz = QuantumCircuit(2)
circuit_crz.crz(1.4, 0, 1)
circuit_crz = transpile(circuit_crz, basis_gates=['u1', 'u2', 'u3', 'cx'], optimization_level=0)
---------------------------------------------------------------------------------------------
u1(0.7) q[1];
cx q[0],q[1];
u1(-0.7) q[1];
cx q[0],q[1];
```
**$u1$ gate:**
```
circuit_u1 = QuantumCircuit(1)
circuit_u1.u1(1.4, 0)
---------------------------------------------------------------------------------------------
u1(1.4) q[0];
```
**$cu1$ gate:**
```
circuit_cu1 = QuantumCircuit(2)
circuit_cu1.cu1(1.4, 0, 1)
circuit_cu1 = transpile(circuit_cu1, basis_gates=['u1', 'u2', 'u3', 'cx'], optimization_level=0)
---------------------------------------------------------------------------------------------
u1(0.7) q[0];
cx q[0],q[1];
u1(-0.7) q[1];
cx q[0],q[1];
u1(0.7) q[1];
```




  [1]: https://quantumcomputing.stackexchange.com/questions/10022/are-mathrmcrz-with-angle-pi-and-mathrmcz-equal-on-the-ibm-quantum-e/10026#10026


So as was discussed in the question $Issue 3$ and in the answer of $Issue 1$ **the control version of this gates are not equivalent to each other**:

$$
CU1\big(\frac{\pi}{2}\big) = 
\begin{pmatrix}
1 & 0 &0 &0 \\
0 & 1 &0 &0 \\
0 & 0 &1 &0 \\
0 & 0 &0 &i 
\end{pmatrix} 
\qquad 
CR_z\big(\frac{\pi}{2}\big) = 
\begin{pmatrix}
1 & 0 &0 &0 \\
0 & 1 &0 &0 \\
0 & 0 &e^{-i \frac{\pi}{4}} &0 \\
0 & 0 &0 &e^{i \frac{\pi}{4}} 
\end{pmatrix}$$

So if we are trying to construct a circuit by applying a control version of some unitary, **the global phase of the unitary shouldn't be neglected**. This scenario is not seldom. For example, in QPE (and thus in HHL) algorithm, we should be careful with the global phase in the unitary whose controlled versions are used in the algorithm.


  [1]: https://quantumcomputing.stackexchange.com/q/11888/9459
  [2]: https://quantumcomputing.stackexchange.com/a/12409/9459

In Qiskit one can get the controlled version of any gate with the method `control`.

For S and T, here is the example:



In [ ]:
from qiskit.circuit.library.standard_gates import SGate, TGate

csgate = SGate().control(1) # the parameter is the amount of control points you want
ctgate = TGate().control(1)

circuit = QuantumCircuit(2)
circuit.append(csgate, [0, 1])
circuit.append(ctgate, [0, 1])
print(circuit)

Now if we want to compare these gates with `cu1` or `crz`, we can use the `Operator` class

In [ ]:
from qiskit.circuit.library.standard_gates import CRZGate, CU1Gate
from math import pi

from qiskit.quantum_info import Operator
print(Operator(CU1Gate(pi/2)) == Operator(csgate))
print(Operator(CRZGate(pi/4)) == Operator(ctgate))

So basically `cu1` and `crz` are different gates and the difference is explained in this [answer][1]. One can express $cS$ and $cT$ gates with `cu1` gate, but it is impossible to express them with `crz` gate.

$$
cS = 
\begin{pmatrix}
1&0&0&0 \\
0&1&0&0 \\
0&0&1&0 \\
0&0&0&i \\
\end{pmatrix} =
cu1(\pi/2)
\qquad
cT = 
\begin{pmatrix}
1&0&0&0 \\
0&1&0&0 \\
0&0&1&0 \\
0&0&0&e^{i \pi/4} \\
\end{pmatrix} =
cu1(\pi/4)
$$

while

$$
cRz(\alpha) = 
\begin{pmatrix}
1&0&0&0 \\
0&1&0&0 \\
0&0&e^{-i \alpha/2}&0 \\
0&0&0&e^{i \alpha/2} \\
\end{pmatrix}
$$

and there is no way to choose such $\alpha$ for $cRz(\alpha)$ to be equal to $cS$ or $cT$.

  [1]: https://quantumcomputing.stackexchange.com/a/13414/9459